In [1]:
device = "cuda"
model_ckpt = "meta-llama/Llama-3.2-1B"

preparation_batch_size = 4 
batch_size = 64

valid_size = 4096
train_size = 10000

In [2]:
# Parameters
model_ckpt = "meta-llama/Llama-3.1-8B"
preparation_batch_size = 2


### Preliminaries

In [3]:
import random
import collections


import transformers
import torch
import tqdm.auto
from torch import Tensor

In [4]:
def sinusoidal_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int,
    max_value: int,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    """
    Encodes a tensor of numbers into a sinusoidal representation, inspired by how absolute positional
    encoding works in transformers.

    The encoding is an evaluation of a sine and cosine function at different frequencies, where the
    frequency is determined by the embedding dimension and the allowed range of the input values.

    >>> sinusoidal_encode(
    ...     torch.tensor([-5, 2, 1, 0]),
    ...     embedding_dim=6,
    ...     min_value=-5,
    ...     max_value=5,
    ... )
    tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
            [ 0.6570,  0.7539, -0.1073, -0.9942,  0.9980,  0.0627],
            [-0.2794,  0.9602,  0.3491, -0.9371,  0.9616,  0.2746],
            [-0.9589,  0.2837,  0.7317, -0.6816,  0.8806,  0.4738]])
    """

    if embedding_dim % 2 != 0 and not use_l2_norm:
        raise ValueError("Embedding dimension must be even")

    if use_l2_norm:
        if embedding_dim % 2 == 0:
            reserved_dim = 2
        else:
            reserved_dim = 1
        embedding_dim -= reserved_dim
    else:
        reserved_dim = 0  # will not be used

    domain = max_value - min_value
    y_shape = x.shape + (embedding_dim,)
    y = torch.zeros(y_shape, device=x.device)
    even_indices = torch.arange(0, embedding_dim, 2)
    log_term = torch.log(torch.tensor(domain)) / embedding_dim
    div_term = torch.exp(even_indices * -log_term)
    x = x - min_value
    values = x.unsqueeze(-1).float() * div_term
    y[..., 0::2] = torch.sin(values)
    y[..., 1::2] = torch.cos(values)

    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserved_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)

    if norm_const is not None:
        y *= norm_const

    return y

def binary_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int | float,
    max_value: int | float,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    y = torch.zeros(x.shape + (embedding_dim,), device=x.device)
    reserve_dim = 0 if not use_l2_norm else 1
    x = x - min_value
    maximum = x.max()
    for i in range(embedding_dim - reserve_dim):
        coeff = 2**i
        if maximum < coeff:
            break
        y[..., -i - 1] = torch.floor(x / coeff) % 2
        x = x - coeff * y[..., -i - 1]
    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserve_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)
    if norm_const is not None:
        y *= norm_const
    return y

### Prepare model and data

In [5]:
model = transformers.AutoModel.from_pretrained(model_ckpt).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})
model = model.half().to(device).eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
all_values = torch.arange(0, 1000)
mask = torch.rand(len(all_values), generator=torch.Generator().manual_seed(0))
train_mask = mask < 0.9
valid_mask = ~train_mask & (mask < 0.95)
test_mask = ~train_mask & ~valid_mask

train_values = all_values[train_mask]
valid_values = all_values[valid_mask]
test_values = all_values[test_mask]

In [7]:
all_inputs = all_values.tolist()
train_values_set = set(train_values.tolist())
valid_values_set = set(valid_values.tolist())
test_values_set = set(test_values.tolist())
        
train_inputs = [x for x in all_inputs if x in train_values_set]
valid_inputs = [x for x in all_inputs if x in valid_values_set]
test_inputs = [x for x in all_inputs if x in test_values_set]

# sanity check
assert set(train_inputs) & set(valid_inputs) == set()
assert set(train_inputs) & set(test_inputs) == set()
assert set(valid_inputs) & set(test_inputs) == set()

random.seed(0)
random.shuffle(train_inputs)
random.shuffle(valid_inputs)
random.shuffle(test_inputs)
train_inputs = train_inputs[:train_size]
valid_inputs = valid_inputs[:valid_size]

In [8]:
len(test_inputs)

55

### Constructing altered natural texts -- with all numbers from pre-defined ranges

In [9]:
# cell loading the input texts
import json
from glob import glob
from tqdm import tqdm

import torch
import datasets
from git import Repo
import os

import itertools


HOME_PATH = "./"

def load_data(genre="food-1", downsample_to=0):
    """
    genre: input , genre of dataset you want to load
    data :  output,

    """
    if genre ==  'food-1':
        directory_path = "./FoodRecipe-ImageCaptioning/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/samsatp/FoodRecipe-ImageCaptioning.git/", "./FoodRecipe-ImageCaptioning/")

        with open(HOME_PATH + directory_path + "data/data_strings_local.json", "r") as fp:
            recipes = json.load(fp)
            #print(recipes)
            concated_data = [' '.join(d) for d in recipes.values()]
            data = concated_data
            print(len(data))

    elif genre == 'food-2':
        reciepe_data2 = datasets.load_dataset("m3hrdadfi/recipe_nlg_lite",trust_remote_code=True) #steps o ingredients
        #train 6118 test 1000
        # ['uid', 'name', 'description', 'link', 'ner', 'ingredients', 'steps']
        data  = reciepe_data2['train']['steps']

    elif genre == 'arthmetic-1':

        metamathqa = datasets.load_dataset("meta-math/MetaMathQA") #original_question
        data = metamathqa['train']['original_question']

    elif genre == 'arthmetic-2':

        drop = datasets.load_dataset("ucinlp/drop") #passage
        data = drop['train']['passage']#['section_id', 'query_id', 'passage', 'question', 'answers_spans']

    elif genre == 'arthmetic-3':
        aquarat = datasets.load_dataset("deepmind/aqua_rat") #['question', 'options', 'rationale', 'correct'] go question or rationale
        data = aquarat['train']['question']

    elif genre == 'technical-1':
        icdatta = datasets.load_dataset("atta00/icd10-codes") #['chapter', 'section', 'category', 'category_code', 'code', 'description']
        data = [f"description: {d} | code: {c}" for d,c in zip(icdatta['train']['description'], icdatta['train']['code'] )] # go for description + code

    elif genre == 'technical-2':
        icdcm = datasets.load_dataset("Gokul-waterlabs/ICD-10-CM")#input+output
        data = [f"Description: {d} | code: {c}" for d,c in zip(icdcm['train']['input'], icdcm['train']['output'] )]

    elif genre == 'datetime-1':

        directory_path = "./TimeLineExtractionDecisionLettersCASE/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/irlabamsterdam/TimeLineExtractionDecisionLettersCASE.git", directory_path)

        data = []
        for file in tqdm(glob(HOME_PATH + directory_path + 'data/txt_files/train/*txt')):
            with open(file, 'r') as fp:
                data.append(fp.read())
    else:
        data="ERROR : Pick a genre from [food-1/2, arthmetic-1/2/3, techincal-1/2, datetime]"
        print(data)
    print("Number of samples in the data loaded:", len(data))
    if downsample_to and len(data) > downsample_to:
        print("Downsampling to %s" % downsample_to)
        data = data[:downsample_to]

    return data

texts = list(itertools.chain(*(load_data(k) for k in ['food-1', 'food-2', 'arthmetic-1', 'arthmetic-2', 'arthmetic-3', 'technical-1', 'technical-2', 'datetime-1'])))
print(len(texts))

719
Number of samples in the data loaded: 719


Repo card metadata block was not found. Setting CardData to empty.


Number of samples in the data loaded: 6118


Number of samples in the data loaded: 395000


Number of samples in the data loaded: 77400


...

Number of samples in the data loaded: 25719


Number of samples in the data loaded: 74044


  0%|                                                                                                                                                                                                                        | 0/50 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 2432.89it/s]

Number of samples in the data loaded: 50
676517


In [10]:
import re


def make_str_input(all_possible_operands: list[int]) -> str:
    selected_text = random.choice(texts)
    text_with_replaced_nums = re.sub(r"\d+", lambda _: str(random.choice(all_possible_operands)), selected_text)
    return text_with_replaced_nums

make_str_input(train_inputs), make_str_input(valid_inputs)

('The population of Port Perry is seven times as many as the population of Wellington. The population of Port Perry is 659 more than the population of Lazy Harbor. If Wellington has a population of 699, how many people live in Port Perry and Lazy Harbor combined?',
 "The Gnollish language consists of 338 words, ``splargh,'' ``glumph,'' and ``amr.''  In a sentence, ``splargh'' cannot come directly before ``glumph''; all other sentences are grammatically correct (including sentences with repeated words).  How many valid 545-word sentences are there in Gnollish?")

### Inference of model's hidden states

In [11]:
num_input_ids = tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]
batch_inputs = tokenizer('In a shower, 801 cm of rain falls. The volume of water that falls on 289.564 hectares of ground is:', return_tensors="pt")
torch.isin(batch_inputs.input_ids, num_input_ids)

tensor([[False, False, False, False, False, False,  True, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True, False,  True, False, False, False, False, False]])

In [12]:
tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]

tensor([   15,    16,    17,    18,    19,    20,    21,    22,    23,    24,
          605,   806,   717,  1032,   975,   868,   845,  1114,   972,   777,
          508,  1691,  1313,  1419,  1187,   914,  1627,  1544,  1591,  1682,
          966,  2148,   843,  1644,  1958,  1758,  1927,  1806,  1987,  2137,
         1272,  3174,  2983,  3391,  2096,  1774,  2790,  2618,  2166,  2491,
         1135,  3971,  4103,  4331,  4370,  2131,  3487,  3226,  2970,  2946,
         1399,  5547,  5538,  5495,  1227,  2397,  2287,  3080,  2614,  3076,
         2031,  6028,  5332,  5958,  5728,  2075,  4767,  2813,  2495,  4643,
         1490,  5932,  6086,  6069,  5833,  5313,  4218,  4044,  2421,  4578,
         1954,  5925,  6083,  6365,  6281,  2721,  4161,  3534,  3264,  1484,
         1041,  4645,  4278,  6889,  6849,  6550,  7461,  7699,  6640,  7743,
         5120,  5037,  7261,  8190,  8011,  7322,  8027,  8546,  8899,  9079,
         4364,  7994,  8259,  4513,  8874,  6549,  9390,  6804, 

In [13]:
import gc
import tqdm

def get_hidden_states(model, str_inputs: list[str], batch_size: int) -> tuple[dict[int, Tensor], Tensor]:
    model.eval()
    num_input_ids = tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]

    nums: list[str] = []
    hidden_states = collections.defaultdict(list)
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size
        for batch_str in tqdm.auto.tqdm(itertools.batched(str_inputs, n=batch_size), total=num_batches):
            batch_inputs = tokenizer(batch_str, return_tensors="pt", padding=True, truncation=True)
            num_pos = torch.isin(batch_inputs.input_ids, num_input_ids)
            hidden_reprs = model(**batch_inputs.to(model.device), output_hidden_states=True).hidden_states
            for layer_idx, hidden_state in enumerate(hidden_reprs):
                hidden_states[layer_idx].extend(hidden_state[num_pos].detach().cpu())
            new_nums = tokenizer.batch_decode(batch_inputs.input_ids[num_pos])
            nums.extend(new_nums)

        hidden_states_stacked = {}
        for k in list(hidden_states.keys()):
            v = hidden_states.pop(k)
            hidden_states_stacked[k] = torch.stack(v)
            del v # explicitly delete to save memory
            gc.collect()  # force garbage collection

    labels = torch.tensor(list(map(int, nums)), device=device)
    return hidden_states_stacked, labels

In [14]:
train_input_texts = [make_str_input(train_inputs) for _ in range(train_size)]
valid_input_texts = [make_str_input(valid_inputs) for _ in range(valid_size)]
test_input_texts = [make_str_input(test_inputs) for _ in range(valid_size)]

train_hidden_states, train_labels = get_hidden_states(model, train_input_texts, preparation_batch_size)
assert train_hidden_states[0].shape[0] == len(train_labels)

valid_hidden_states, valid_labels = get_hidden_states(model, valid_input_texts, preparation_batch_size)
assert valid_hidden_states[0].shape[0] == len(valid_labels)

test_hidden_states, test_labels = get_hidden_states(model, test_input_texts, preparation_batch_size)
assert test_hidden_states[0].shape[0] == len(test_labels)


  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/2048 [00:00<?, ?it/s]

  0%|          | 0/2048 [00:00<?, ?it/s]

In [15]:
# sum(((train_hidden_states[0] == valid_hidden_states[0][i]).all(dim=1).any() for i in range(valid_size)))

### Probing

In [16]:
class ClassifierProbe(torch.nn.Module):
    basis: torch.Tensor

    def __init__(self, emb_dim: int, hidden_dim: int, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.basis_to_latent = torch.nn.Linear(self.basis.shape[-1], hidden_dim, bias=True)
        self.basis = self.basis.to(device)
        self.heldout_mask: torch.nn.Buffer
        # self.register_buffer("basis", self.basis)
        self.register_buffer("heldout_mask", heldout_mask)
    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        latent_choices = self.basis_to_latent(self.basis)
        logits = latent_x @ latent_choices.T
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = float("-inf")
        return logits

In [17]:
class SinProbeOld(ClassifierProbe):

    def __init__(self, *args, **kwargs):
        self.basis = sinusoidal_encode(torch.arange(1000), min_value=0, max_value=1000,
                                       embedding_dim=train_hidden_states[0].shape[-1])
        super().__init__(*args, **kwargs)

class BinProbe(ClassifierProbe):

    def __init__(self, *args, **kwargs):
        self.basis = binary_encode(torch.arange(1000), min_value=0, max_value=1000, embedding_dim=10).to(device)
        super().__init__(*args, **kwargs)


In [18]:
class SinProbeNew(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, choices: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.freqs = torch.nn.Parameter(torch.linspace(1/(choices.max() - choices.min()), 0.5, steps=hidden_dim))
        self.phases = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.amplitudes = torch.nn.Parameter(torch.ones(hidden_dim) * 0.0001)
        # self.accels = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.hidden_dim = hidden_dim
        self.heldout_mask: torch.nn.Buffer
        self.choices: torch.nn.Buffer
        self.register_buffer("heldout_mask", heldout_mask)
        self.register_buffer("choices", choices)

    def get_waves(self) -> Tensor:
        # USE THIS FORMULA
        waves = torch.sin(
            self.phases.unsqueeze(1)
            + (2 * torch.pi * self.freqs.unsqueeze(1) * self.choices.unsqueeze(0))
            # + (2 * torch.pi * self.accels.unsqueeze(1) * torch.log(self.choices.unsqueeze(0) + 1e-4))
        )
        # sort by frequency
        # waves = waves[torch.argsort(self.freqs.abs()), :]
        # assert waves.shape == (self.hidden_dim, len(self.choices))
        return waves * self.amplitudes.unsqueeze(1)

    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        waves = self.get_waves()
        logits = latent_x @ waves

        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = -torch.inf
        return logits

In [19]:
# Held-one-out: Training on all-minus-one

torch.manual_seed(0)
rng = torch.Generator().manual_seed(0)
rng_py = random.Random(0)


assert list(train_hidden_states.keys()) == list(range(len(train_hidden_states)))
train_hidden_states_tensor = torch.stack(list(train_hidden_states.values()), dim=0)

heldout_probes = {}
heldout_histories = []

test_accuracies = {"sin": {}, "sin_old": {}, "bin": {}, "lin": {}, "log": {}}

if device != "cpu":
    torch.set_num_threads(8)


for heldout_layer_idx in range(len(train_hidden_states)):
    probe: torch.nn.Module
    for probe_name, probe in {
            "sin": SinProbeNew(
                        emb_dim=train_hidden_states[0].shape[-1],
                        hidden_dim=500,
                        choices=torch.arange(1000),
                        heldout_mask=test_mask,
                    ).to(device),
            "sin_old": SinProbeOld(emb_dim=train_hidden_states[0].shape[-1],
                        hidden_dim=100,
                        heldout_mask=test_mask,
                    ).to(device),
            "bin": BinProbe(
                        emb_dim=train_hidden_states[0].shape[-1],
                        hidden_dim=100,
                        heldout_mask=test_mask,
                    ).to(device),
                }.items():
        
        torch.manual_seed(0)

        if isinstance(probe, SinProbeNew):
            reg_params = [probe.amplitudes, *probe.emb_to_latent.parameters()]
            noreg_params = [probe.freqs, probe.phases]
        else:
            reg_params = []
            noreg_params = list(probe.parameters())

        optimizer = torch.optim.Adam(
            [
                {"params": noreg_params, "weight_decay": 0.0},
                {"params": reg_params, "weight_decay": 1e-3},
            ],
            lr=1e-4,
        )
        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=15000)

        train_layers = [i for i in range(len(train_hidden_states)) if i != heldout_layer_idx]
        train_layers_tensor = torch.tensor(train_layers)

        best_val_acc = -1
        best_ckpt = probe.state_dict()

        layer_idcs = torch.tensor(random.choices(train_layers, k=batch_size))
        minibatch_idcs = torch.randint(len(train_labels), size=(batch_size,), generator=rng)
        next_x = train_hidden_states_tensor[layer_idcs, minibatch_idcs].to(device, dtype=torch.float32, non_blocking=True)
        next_y = train_labels[minibatch_idcs].to(device, non_blocking=True)

        print("HELDOUT LAYER:", heldout_layer_idx)
        for step in range(30000+1):
            probe.train()
            optimizer.zero_grad()

            x, y = next_x, next_y
            torch.cuda.synchronize() # ensure the current batch is on the device

            # asynchronously prefetch the next batch on the device
            random_indices = torch.randint(0, len(train_layers_tensor), (batch_size,), generator=rng)
            next_layer_idcs = train_layers_tensor[random_indices]
            next_minibatch_idcs = torch.randint(len(train_labels), size=(batch_size,), generator=rng)
            next_x = train_hidden_states_tensor[next_layer_idcs, next_minibatch_idcs].to(device, dtype=torch.float32, non_blocking=True)
            next_y = train_labels[next_minibatch_idcs].to(device, non_blocking=True)

            train_logits = probe(x, holdout_eval_tokens=True)
            loss = torch.nn.functional.cross_entropy(train_logits, y)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
        
            if step % 1000 == 0:
                probe.eval()
                valid_accs = []
                with torch.no_grad():
                    print(f"{step=:<5}", end="  ")
                    for layer_idx in range(0, len(train_hidden_states)):
                        valid_logits = probe(valid_hidden_states[layer_idx].to(device, dtype=torch.float32), holdout_eval_tokens=False)
                        valid_acc = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                        valid_accs.append(valid_acc)
                        heldout_histories.append({"heldout_layer": heldout_layer_idx, "step": step, "eval_layer": layer_idx, "valid_acc": valid_acc})
                        acc_out = f"{valid_acc:>6.1%}"
                        if layer_idx not in train_layers:
                            print('\033[94m' + acc_out + '\033[0m', end=" ")
                        else:
                            print(acc_out, end=" ")
                    print()
                    if valid_accs[heldout_layer_idx] > best_val_acc:
                        best_val_acc = valid_accs[heldout_layer_idx]
                        best_ckpt = probe.state_dict()

        probe.load_state_dict(best_ckpt)
        probe.eval()
        with torch.no_grad():
            test_logits = probe(test_hidden_states[heldout_layer_idx].float().to(device), holdout_eval_tokens=False)
            test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean().item()
        test_accuracies[probe_name][heldout_layer_idx] = test_accuracy
        print(f"->  {probe_name}  heldout layer idx: {heldout_layer_idx:<3}, best valid accuracy: {best_val_acc:.2f}, test accuracy: {test_accuracy:.2f}", flush=True)

HELDOUT LAYER: 0
step=0      

  0.0%   0.0% 

  0.0%   0.0% 

  0.1%   0.2% 

  0.1%   0.0% 

...

  4.6%   4.4% 

  4.3%   4.1% 

  4.0%   4.2% 

  4.0%   0.4% 


->  bin  heldout layer idx: 32 , best valid accuracy: 0.00, test accuracy: 0.00


In [20]:
test_accuracies

{'sin': {0: 0.4908944070339203,
  1: 1.0,
  2: 0.9991490244865417,
  3: 0.9978299736976624,
  4: 0.999744713306427,
  5: 0.998383104801178,
  6: 0.9985533356666565,
  7: 0.9982129335403442,
  8: 0.9977874159812927,
  9: 0.9985958933830261,
  10: 0.9988086223602295,
  11: 0.9988086223602295,
  12: 0.9969789981842041,
  13: 0.9966385960578918,
  14: 0.9730661511421204,
  15: 0.998978853225708,
  16: 0.9991064667701721,
  17: 0.9985533356666565,
  18: 0.998978853225708,
  19: 0.997872531414032,
  20: 0.9992766976356506,
  21: 0.9988086223602295,
  22: 0.9975321292877197,
  23: 0.999234139919281,
  24: 0.9975321292877197,
  25: 0.987107515335083,
  26: 0.996510922908783,
  27: 0.9975321292877197,
  28: 0.9959152340888977,
  29: 0.9945962429046631,
  30: 0.9840864539146423,
  31: 0.98310786485672,
  32: 0.05871840938925743},
 'sin_old': {0: 0.05625053122639656,
  1: 0.9229001998901367,
  2: 0.966002881526947,
  3: 0.9689813852310181,
  4: 0.9655774235725403,
  5: 0.9850225448608398,
  6: 0.

In [21]:
def solve_linear_layer(x: Tensor, y: Tensor) -> torch.nn.Linear:
    if y.ndim == 1:
        y = y.unsqueeze(-1)
    if not y.is_floating_point():
        y = y.float()
   
    lin = torch.nn.Linear(x.shape[-1], y.shape[-1], device=x.device)
    x_aug = torch.cat([x, torch.ones(len(x), 1, device=x.device)], dim=1)
    coeffs = torch.linalg.lstsq(x_aug, y).solution
    w, b = coeffs[:-1], coeffs[-1]
    with torch.no_grad():
        lin.weight[:] = w.T
        lin.bias[:] = b
    return lin

In [22]:
for layer_idx in range(len(train_hidden_states)):
    lin_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.to(device),
    )
    log_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.log1p().to(device),
    )
    lin_test_pred = lin_probe(test_hidden_states[layer_idx].float().to(device)).flatten().round().int()
    lin_test_accuracy = (lin_test_pred == test_labels).float().mean().item()
    
    log_test_pred = log_probe(test_hidden_states[layer_idx].float().to(device)).flatten().exp().add(1).round().int()
    log_test_accuracy = (log_test_pred == test_labels).float().mean().item()
    
    test_accuracies["lin"][layer_idx] = lin_test_accuracy
    test_accuracies["log"][layer_idx] = log_test_accuracy

    print(f"layer idx: {layer_idx:<3}, linear probe acc: {lin_test_accuracy:.2f}, log probe acc: {log_test_accuracy:.2f}")

layer idx: 0  , linear probe acc: 0.00, log probe acc: 0.00


layer idx: 1  , linear probe acc: 0.01, log probe acc: 0.01


layer idx: 2  , linear probe acc: 0.02, log probe acc: 0.01


layer idx: 3  , linear probe acc: 0.02, log probe acc: 0.01


layer idx: 4  , linear probe acc: 0.02, log probe acc: 0.01


...

layer idx: 28 , linear probe acc: 0.01, log probe acc: 0.01


layer idx: 29 , linear probe acc: 0.01, log probe acc: 0.01


layer idx: 30 , linear probe acc: 0.01, log probe acc: 0.01


layer idx: 31 , linear probe acc: 0.01, log probe acc: 0.01


layer idx: 32 , linear probe acc: 0.01, log probe acc: 0.00


In [23]:
test_accuracies

{'sin': {0: 0.4908944070339203,
  1: 1.0,
  2: 0.9991490244865417,
  3: 0.9978299736976624,
  4: 0.999744713306427,
  5: 0.998383104801178,
  6: 0.9985533356666565,
  7: 0.9982129335403442,
  8: 0.9977874159812927,
  9: 0.9985958933830261,
  10: 0.9988086223602295,
  11: 0.9988086223602295,
  12: 0.9969789981842041,
  13: 0.9966385960578918,
  14: 0.9730661511421204,
  15: 0.998978853225708,
  16: 0.9991064667701721,
  17: 0.9985533356666565,
  18: 0.998978853225708,
  19: 0.997872531414032,
  20: 0.9992766976356506,
  21: 0.9988086223602295,
  22: 0.9975321292877197,
  23: 0.999234139919281,
  24: 0.9975321292877197,
  25: 0.987107515335083,
  26: 0.996510922908783,
  27: 0.9975321292877197,
  28: 0.9959152340888977,
  29: 0.9945962429046631,
  30: 0.9840864539146423,
  31: 0.98310786485672,
  32: 0.05871840938925743},
 'sin_old': {0: 0.05625053122639656,
  1: 0.9229001998901367,
  2: 0.966002881526947,
  3: 0.9689813852310181,
  4: 0.9655774235725403,
  5: 0.9850225448608398,
  6: 0.

In [24]:
for name, accs in test_accuracies.items():
    print(f"{name} accs: | " + " | ".join([f"{x:.0%}" for layer, x in sorted(accs.items())]) + " |")

sin accs: | 49% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 97% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 100% | 99% | 100% | 100% | 100% | 99% | 98% | 98% | 6% |
sin_old accs: | 6% | 92% | 97% | 97% | 97% | 99% | 97% | 98% | 97% | 97% | 97% | 96% | 97% | 98% | 99% | 99% | 99% | 99% | 99% | 98% | 98% | 98% | 97% | 97% | 98% | 97% | 95% | 96% | 95% | 93% | 92% | 87% | 5% |
bin accs: | 0% | 3% | 2% | 3% | 3% | 2% | 2% | 2% | 2% | 2% | 2% | 3% | 2% | 2% | 1% | 3% | 5% | 5% | 5% | 5% | 5% | 5% | 4% | 5% | 4% | 5% | 5% | 6% | 5% | 4% | 4% | 4% | 0% |
lin accs: | 0% | 1% | 2% | 2% | 2% | 2% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% |
log accs: | 0% | 1% | 1% | 1% | 1% | 2% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 0% |
